In [1]:
!pip install rank_bm25
!pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 16.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!tar -xvf /content/drive/MyDrive/anlp-spring2026-hw2.tar

anlp-spring2026-hw2/
anlp-spring2026-hw2/.vscode/
anlp-spring2026-hw2/.vscode/settings.json
anlp-spring2026-hw2/data_process/
anlp-spring2026-hw2/data_process/baseline_data/
anlp-spring2026-hw2/data_process/baseline_data/2025-operating-budget.pdf
anlp-spring2026-hw2/data_process/baseline_data/Andrew Carnegie - Wikipedia.htm
anlp-spring2026-hw2/data_process/baseline_data/Andrew Carnegie - Wikipedia_files/
anlp-spring2026-hw2/data_process/baseline_data/Andrew Carnegie - Wikipedia_files/120px-Macomb_Public_Library.JPG
anlp-spring2026-hw2/data_process/baseline_data/Andrew Carnegie - Wikipedia_files/120px-Yorkville_Library.jpg
anlp-spring2026-hw2/data_process/baseline_data/Andrew Carnegie - Wikipedia_files/150px-Andrew_Carnegie_signature.svg.png
anlp-spring2026-hw2/data_process/baseline_data/Andrew Carnegie - Wikipedia_files/170px-Andrew_Carnegie,_Vanity_Fair,_1903-10-29.jpg
anlp-spring2026-hw2/data_process/baseline_data/Andrew Carnegie - Wikipedia_files/180px-Macomb_Public_Library.JPG
anlp

In [9]:
import pickle
import json
import re
from pathlib import Path
from tqdm import tqdm
from rank_bm25 import BM25Okapi

In [10]:
def clean_text(text):
    # 替换换行/tab为空格
    text = re.sub(r"[\n\t\r]+", " ", text)

    # 多空格压缩
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [11]:
folder = Path(r"/content/anlp-spring2026-hw2/hybrid_retrieval/data")

# 获取所有 .json 文件
htm_files = list(folder.glob("*.json"))

corpus = []
# 保证数据顺序不要出错
for i, file in tqdm(enumerate(htm_files), total=len(htm_files)):
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
        for chunk in data:
            text = clean_text(
                f"""
                Title: {chunk["title"]}

                Section: {chunk["section"]}

                Content:{chunk["text"]}
                """
            )
            corpus.append(text)

100%|██████████| 89/89 [00:00<00:00, 110.91it/s]


In [12]:
tokenized_corpus = [doc.split() for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
with open("bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)

In [ ]:
bm25 = pickle.load(open("bm25.pkl", "rb"))

In [13]:
query = "Who is Pittsburgh named after? "

In [14]:
bm25_scores = bm25.get_scores(
    query.split()
)

idx = bm25_scores.argsort()[::-1]
top_k = 30

In [15]:
top_idx = idx[:top_k]

In [16]:
sentences = [corpus[i] for i in top_idx]
sentences

['Title: The Pittsburgh Cultural Trust Section: Staff Picks: Meet the Women Who Inspire Us Content: For Women’s History Month, we asked a some of our female-identifying staff member to reflect on women in art who inspire them... Read More',
 'Title: About Carnegie Mellon _ Carnegie Mellon University Section: Who We Are and What We Do Content: CMU fosters a community of fearless trailblazers united by a shared passion for innovation and excellence. Learn about our strategic partnerships and global presence.',
 "Title: Scotch'n'Soda Theatre - Wikipedia Section: Alumni Clan Content: Scotch'n'Soda is currently developing an Alumni Clan, which is devoted to keeping all past members of the organization in touch and organizing reunion events. It also presents the Buzz Blair Award for the Performing Arts, named after founding member Leonard 'Buzz' Blair, and given annually during Carnegie Mellon's homecoming festivities.",
 'Title: Pittsburgh - Wikipedia Section: Food Content: Pittsburgh is kn

In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("ashritharai/all-MiniLM-L6-v2")


embeddings = model.encode(sentences)

import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

scores, indices = index.search(query_embedding, k=3)

print(indices)
print(scores)

[[15  3  8]]
[[0.5754243  0.5556158  0.54416466]]


In [6]:
indices = [[15,  3,  8]]

In [17]:
prompt = f"Question: {query}\n\n"

prompt += "References:\n"

sentences = [sentences[i] for i in indices[0]]

for i, doc in enumerate(sentences):

    prompt += f"[{i+1}] {doc}\n\n"

prompt += "Answer:"

In [22]:
from huggingface_hub import login
login()

In [29]:
import gc
gc.collect()

6383

In [32]:
!nvidia-smi
# 查找所有包含"python"的进程
!ps -ef | grep python

Fri May 15 09:25:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             30W /   70W |   14909MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!kill -9 566

In [18]:
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")
messages = [
    {"role": "user", "content": prompt},
]
pipe(messages)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user',
    'content': 'Question: Who is Pittsburgh named after? \n\nReferences:\n[1] Title: Pittsburgh - Wikipedia Section: 18th century Content:l John Forbes finally took the forks in 1758. He began construction on Fort Pitt , named after William Pitt the Elder , while the settlement was named "Pittsborough". During Pontiac\'s War , a loose confederation of Native American tribes laid siege to Fort Pitt in 1763; the siege was eventually lifted after Colonel Henry Bouquet defeated a portion of the besieging force at the Battle of Bushy Run . Bouquet strengthened the defenses of Fort Pitt the next year. During this period, the powerful nations of the Iroquois Confederacy , based in New York, had maintained control of much of the Ohio Valley as hunting grounds by right of conquest after defeating other tribes. By the terms of the 1768 Treaty of Fort Stanwix , the Penns were allowed to purchase the modern region from the Iroquois . A 1769 survey referenced t